# Lab 23-08 · Heating a Phase-Change Cell and Flipping Magnets

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import emem


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · ngspice

*Predict* the peak rise, the quench time and the SET plateau.

In [2]:
sim = ngspice("pcm_thermal.cir")
print(f"RESET peak rise: ngspice {sim['dtreset']:.1f} K, model {emem.thermal_rise(emem.joule_power(200e-6, 2e3), 7.5e6):.1f} K")
print(f"cooling 500 K -> 125 K: ngspice {sim['tquench']*1e9:.3f} ns, model {emem.cooling_time(2e-9, 500, 125)*1e9:.3f} ns")
print(f"SET plateau: ngspice {sim['dtset']:.1f} K")

RESET peak rise: ngspice 600.0 K, model 600.0 K
cooling 500 K -> 125 K: ngspice 2.773 ns, model 2.773 ns
SET plateau: ngspice 300.0 K


## Part 2 · MRAM thermal stability

*Predict* how much more $\Delta$ a 1,000-fold larger array needs.

In [3]:
years10 = 10 * 365.25 * 86400
n = np.logspace(3, 12, 100)
plt.figure()
for t_c in (25, 85, 150):
    plt.semilogx(n, [emem.delta_at(emem.delta_required(x, years10), t_c, 25) for x in n], label=f"{t_c} °C")
plt.xlabel("bits in the array"); plt.ylabel("room-temperature Delta needed"); plt.legend(); plt.show()
d = emem.delta_required(2**30, years10)
print(f"1 Gb, 10 years: Delta = {d:.2f} at operating temperature; {emem.delta_at(d, 85, 25):.2f} at 25 °C for 85 °C operation")

1 Gb, 10 years: Delta = 61.09 at operating temperature; 73.38 at 25 °C for 85 °C operation


## Part 3 · Watching bits flip

*Predict* the number of flips among 10,000 bits with $\Delta$ = 25 in one second.

In [4]:
rng = np.random.default_rng(2308)
n_bits, delta, t = 10_000, 25, 1.0
tau = emem.retention_time(delta)
flips = int((rng.exponential(tau, n_bits) < t).sum())
print(f"simulated flips: {flips}; expected N t f0 exp(-Delta) = {n_bits * t * 1e9 * math.exp(-delta):.1f}")

simulated flips: 132; expected N t f0 exp(-Delta) = 138.9
